In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

## Sobre o dataset - Retailrocket
É um dataset de ecommerce baseado em eventos com informações sobre os itens (propriedades e categorias) que foram acessados.

##### 3 arquivos de dados principais:
- events.csv:
    - timestamp: timestamp (unix timestamp em milisegundos) do snapshot do evento;
    - visitorid: id unico do visitante;
    - event: tipo do evento: view, addtocart, transaction;
    - itemid: id unico do item;
    - transactionid: id unico da transacao, caso tenha na sessão.

- item_properties.сsv:
    - timestamp: timestamp (unix timestamp em milisegundos) do snapshot do evento;
    - itemid: id unico do item;
    - property: propriedade do item;
    - value: propriedade valor do item.

- category_tree.сsv:
    - categoryid: identificador unico da categoria;
    - parentid: identificador unico do "pai" da categoria, se for vazio nao existe "pai".


## 1. conhecendo o dataset - events

In [ ]:
df_events = pd.read_csv("../data/interim/events.csv")

df_events.head()

In [ ]:
df_events['timestamp'] = pd.to_datetime(df_events['timestamp'], unit='ms')

df_events['data'] = df_events['timestamp'].dt.date
df_events['hora'] = df_events['timestamp'].dt.hour
df_events['dia_semana'] = df_events['timestamp'].dt.day_name()

In [ ]:
print(df_events.shape)

In [ ]:
print(df_events.isnull().sum())

In [ ]:
print(df_events.info())

In [ ]:
eventos = df_events['event'].value_counts()

eventos.plot(
    kind='bar'
)

plt.title('Distribuição dos Eventos')
plt.show()

In [ ]:
eventos_dia = (
    df_events.groupby('data')
      .size()
)

eventos_dia.plot(
    figsize=(15,5)
)

plt.title("Eventos ao longo do tempo")
plt.show()

In [ ]:
ordem = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday'
]

sns.countplot(
    data=df_events,
    x='dia_semana',
    order=ordem
)

plt.title("Distribuição dos eventos nos dias da semana")
plt.xticks(rotation=45)
plt.show()

In [ ]:
top_items = (
    df_events[df_events['event']=='view']
      ['itemid']
      .value_counts()
      .head(10)
)

top_items.plot(
    kind='bar'
)

plt.title('Top 10 Produtos Mais Visualizados')
plt.show()

In [ ]:
comprados = (
    df_events[df_events['event']=='transaction']
      ['itemid']
      .value_counts()
      .head(10)
)

comprados.plot(
    kind='bar'
)

plt.title('Top 10 Produtos Comprados')
plt.show()

In [ ]:
interacoes_usuario = (
    df_events.groupby('visitorid')
      .size()
)

interacoes_usuario.describe()

In [ ]:
views = (df_events['event'] == 'view').sum()

cart = (df_events['event'] == 'addtocart').sum()

trans = (df_events['event'] == 'transaction').sum()


funil = pd.DataFrame({
    'etapa':['view','addtocart','transaction'],
    'quantidade':[views, cart, trans]
})

sns.barplot(
    data=funil,
    x='etapa',
    y='quantidade'
)

plt.title('Funil de Conversão')
plt.show()

In [ ]:
df_events['peso'] = df_events['event'].map({
    'view':1,
    'addtocart':3,
    'transaction':5
})

interacoes = (
    df_events.groupby(
        ['visitorid','itemid']
    )['peso']
    .sum()
    .reset_index()
)

display(interacoes)

## 1. conhecendo o dataset - item propertie

In [ ]:
df_prop_1 = pd.read_csv("../data/interim/item_properties_part1.csv")
df_prop_2 = pd.read_csv("../data/interim/item_properties_part2.csv")

df_prop = pd.concat([df_prop_1,df_prop_2], ignore_index=True)

df_prop.head()

In [ ]:
df_prop['property'].value_counts().head(20)

In [ ]:
props_por_item = (
    df_prop.groupby('itemid')
           .size()
)

In [ ]:
props_por_item.hist(bins=50)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RetailRocket")
    .getOrCreate()
)

In [ ]:
df_events = spark.read.csv(
    "../data/interim/events.csv",
    header=True,
    inferSchema=True
)

df_prop_1 = spark.read.csv(
    "../data/interim/item_properties_part1.csv",
    header=True,
    inferSchema=True
)

df_prop_2 = spark.read.csv(
    "../data/interim/item_properties_part2.csv",
    header=True,
    inferSchema=True
)


df_prop = df_prop_1.unionByName(df_prop_2)

df_prop.groupBy(
    "property"

).count().show()

In [ ]:
# Pegar a última versão de cada propriedade
from pyspark.sql import Window
import pyspark.sql.functions as F

window_spec = Window.partitionBy("itemid", "property").orderBy("timestamp").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

df_prop_last = (
    df_prop
    # Aplica a função last sobre a janela para manter a ordem correta do timestamp
    .withColumn("last_value", F.last("property").over(window_spec))
    # Agrupa para remover as duplicadas e trazer o resultado final
    .groupBy("itemid", "property")
    .agg(F.last("property").alias("property"), F.last("timestamp").alias("timestamp"))
)


In [ ]:
props = df_prop_last[
    df_prop_last['property']
    .isin([
        'categoryid',
        'available'
    ])
]

In [ ]:
props.show(10)

In [ ]:
df_items = (
    props
    # 1. O 'index' vira o groupBy
    .groupBy("itemid")
    # 2. O 'columns' vira o argumento do pivot
    .pivot("property")
    # 3. O 'values' entra dentro de uma função de agregação (ex: first ou max)
    .agg(F.first("value"))
)

In [ ]:
df_eda = df_events.join(
    df_items,
    on='itemid',
    how='left'
)

In [ ]:
df_eda.show()

In [ ]:
import seaborn as sns

In [ ]:
views_categoria = (
    df_eda
    .filter("event='view'")
    .groupBy("categoryid")
    .count()
    .orderBy("count", ascending=False)
    .limit(20)
    .toPandas()
)

sns.barplot(
    data=views_categoria,
    y="categoryid",
    x="count"
)

plt.title("Top 20 Categorias Mais Visualizadas")
plt.show()

In [ ]:
users = (
    df_eda
    .groupBy("visitorid")
    .count()
    .toPandas()
)

sns.histplot(
    users["count"],
    bins=100
)

plt.xscale("log")
plt.title("Interações por Usuário")
plt.show()


In [ ]:
from pyspark.sql.functions import from_unixtime

df_temp = (
    df_eda.withColumn(
        "data",
        from_unixtime(
            F.col("timestamp")/1000
        ).cast("timestamp")
    )
)

timeline = (
    df_temp
    .groupBy(
        F.to_date("data")
    )
    .count()
    .orderBy("to_date(data)")
    .toPandas()
)

plt.figure(figsize=(15,5))

plt.plot(
    timeline.iloc[:,0],
    timeline["count"]
)

plt.title("Eventos ao Longo do Tempo")
plt.show()

In [ ]:
produto_pop = (
    df_eda
    .groupBy("itemid")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
)

plt.figure(figsize=(12,5))

plt.plot(
    range(len(produto_pop)),
    produto_pop["count"]
)

plt.yscale("log")

plt.title(
    "Long Tail dos Produtos"
)

plt.xlabel("Ranking do Produto")
plt.ylabel("Quantidade de Interações")
plt.show()

In [ ]:
spark.stop